# E3 · 100% recall with a random forest: the rebalancing done before the split

**Type**: Find the bug · **Level**: ●●● challenge · **Time**: about 45 minutes · **extended path**

## Objective

Find out why a random forest on a problem with 7% positives gets recall, precision and AUC close to 1: the colleague **duplicated the positives to balance the classes before** splitting training and test, so
the same examples are in both. Fix it (split first, rebalancing only on the training set) and measure the true performance.

## Dataset

`generate()` (given): 3000 examples, 10 features (4 informative), about 7% positives (`df` with a `y` column). Model: `RandomForestClassifier(200, random_state=0)`.

## Educational purpose

Recognize rebalancing leakage, one of the most common in production: any transformation that **duplicates or synthesizes** rows (oversampling, SMOTE, augmentation) must happen after the split and only on the training set.
An identical copy in the test set is an example the model has already seen.

## Task

1. Run the colleague's cell: `bug_recall`, `bug_precision`, `bug_auc` are almost perfect. Be suspicious.
2. Compute `test_duplicates_share`: the fraction of the positives of the colleague's test set (`Xt_bug`, `yt_bug`) that appear identical (same values in all the features) in `Xa_bug`.
3. Fix it: split **first** (stratified `train_test_split` 70/30, `random_state=0`) the original data, duplicate the positives **only in the training set** (`sample(..., replace=True, random_state=0)`), train and evaluate on the original test set. Save `ok_recall`, `ok_precision`, `ok_auc` and `ok_ap` (average precision).
4. Compare with the solution without duplicates: forest with `class_weight="balanced"` on the non-duplicated training set. Save `weights_ap` and `weights_recall`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split


def generate():
    X, y = make_classification(n_samples=3000, n_features=10, n_informative=4, n_redundant=0, weights=[0.94], class_sep=0.8, flip_y=0.02, random_state=0)
    df = pd.DataFrame(X, columns=[f"x{i}" for i in range(10)])
    df["y"] = y
    return df


df = generate()
rf = lambda **kw: RandomForestClassifier(200, random_state=0, n_jobs=-1, **kw)
print("fraction of positives:", df["y"].mean().round(3))

## Your work

### The colleague's code

In [ ]:
majority, minority = df[df["y"] == 0], df[df["y"] == 1]
balanced = pd.concat([majority, minority.sample(len(majority), replace=True, random_state=0)], ignore_index=True)     # 50% / 50%
Xa_bug, Xt_bug, ya_bug, yt_bug = train_test_split(balanced.drop(columns="y"), balanced["y"], test_size=0.3, stratify=balanced["y"], random_state=0)
m_bug = rf().fit(Xa_bug, ya_bug)
pred_bug, score_bug = m_bug.predict(Xt_bug), m_bug.predict_proba(Xt_bug)[:, 1]
bug_recall, bug_precision, bug_auc = recall_score(yt_bug, pred_bug), precision_score(yt_bug, pred_bug), roc_auc_score(yt_bug, score_bug)
print(f"recall {bug_recall:.3f} | precision {bug_precision:.3f} | AUC {bug_auc:.3f}")

### Your diagnosis and the fix

In [ ]:
test_duplicates_share = None

In [ ]:
ok_recall = None
ok_precision = None
ok_auc = None
ok_ap = None

In [ ]:
weights_ap = None
weights_recall = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Duplicates: `Xa_bug.drop_duplicates()` and `Xt_bug[yt_bug == 1].merge(Xa_bug.drop_duplicates(), how="left", indicator=True)["_merge"].eq("both").mean()`. Fix: `tr = pd.concat([X_tr, y_tr], axis=1)`; then `minority.sample(len(majority), replace=True, random_state=0)` **only on `tr`**.'):
    check(bug_recall > 0.95 and bug_precision > 0.95 and bug_auc > 0.98, "the colleague's numbers are too good to be true", "don't modify the colleague's code")
    check(test_duplicates_share is not None and test_duplicates_share > 0.9, "over 90% of the positives of the test set appear identical in the training set", "merge between Xt_bug (positives only) and Xa_bug.drop_duplicates()")
    check(ok_recall < 0.4 and 0.75 < ok_auc < 0.92, "on the true test set: low recall (below 0.4) and AUC 0.75-0.92", "the test set must have the original prevalence (7%) and no duplicates")
    check(ok_ap < 0.6, "average precision below 0.6: the problem is hard", "average_precision_score on the original test set")
    check(weights_ap is not None and abs(weights_ap - ok_ap) < 0.15 and weights_recall < 0.4, "class_weight='balanced' gives comparable results, with no duplicates and no leakage", "RandomForestClassifier(class_weight='balanced') on the original training set")
done()

---
The commented solution is in `solutions/E_find_the_bug/E3_oversampling_before_the_split.ipynb`. Open it after you pass the check, or after you have really tried.